# Can we predict whether a student passes the closing quiz (`passed_quiz`)?

Plan, following `checklist.md`:
- The data has no date/time column, so there is no "later rows" ordering. `student_key` repeats across rows (about 7 sessions per student), so the test set and every CV fold hold **whole students** (grouped splits).
- Features: `enrol_year`, `school_code`, `prior_score`, `session_minutes`, `hints_used`. `student_key` is an identifier and is dropped.
- Timing caveat: `columns.csv` does not say when `session_minutes` and `hints_used` are recorded. They describe the session that precedes the closing quiz, so I treat them as known when the quiz starts. If the model must predict before the session begins, they would not be usable; this is flagged in the conclusions.
- `prior_score` is described as a placement-test score "as stored on the session record". It varies within a student, so it is not a single fixed placement score. I keep it, but flag it.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score, log_loss
from sklearn.base import clone

SEED=0
df=pd.read_csv('data.csv')
TARGET='passed_quiz'; ID='student_key'
print(df.shape, df[ID].nunique(), 'students; pass rate', round(df[TARGET].mean(),3))

## 1. Set aside the test set (whole students, 20%)
No test-set rows are examined or scored until step 6.

In [ ]:
gss=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=SEED)
tr_idx,te_idx=next(gss.split(df,groups=df[ID]))
train,test=df.iloc[tr_idx].reset_index(drop=True),df.iloc[te_idx].reset_index(drop=True)
assert set(train[ID]).isdisjoint(test[ID])
print(len(train),len(test))
FEATS=['enrol_year','school_code','prior_score','session_minutes','hints_used']
Xtr,ytr,gtr=train[FEATS],train[TARGET],train[ID]
Xte,yte=test[FEATS],test[TARGET]
del test, df

## 2. Look at the training rows only

In [ ]:
print(train[FEATS+[TARGET]].describe().T)
print(train.groupby(pd.qcut(train.hints_used,4,duplicates='drop'),observed=True)[TARGET].mean())
print(train[FEATS+[TARGET]].corr()[TARGET])

## 3. Candidate models (named before any scoring)
1. **baseline**: predict the training pass rate for everyone.
2. **logreg**: standardised numeric features + one-hot `school_code`, logistic regression.
3. **logreg_nocode**: same without `school_code`.
4. **hgb**: histogram gradient boosting on the raw features, with `school_code` as a numeric column.
5. **hgb_small**: the same with shallower trees and stronger regularisation.

Selection metric: ROC AUC (mean over folds). Decision threshold: chosen on pooled out-of-fold predictions of the winner by maximising balanced accuracy. Preprocessing sits inside each pipeline, so it is fit within each fold.

In [ ]:
num=['enrol_year','prior_score','session_minutes','hints_used']
def lr(cols,cat):
    tf=[('n',StandardScaler(),[c for c in cols if c in num])]
    if cat: tf.append(('c',OneHotEncoder(handle_unknown='ignore'),['school_code']))
    return Pipeline([('p',ColumnTransformer(tf)),('m',LogisticRegression(max_iter=1000))])
MODELS={
 'baseline':Pipeline([('m',DummyClassifier(strategy='prior'))]),
 'logreg':lr(FEATS,True),
 'logreg_nocode':lr([c for c in FEATS if c!='school_code'],False),
 'hgb':Pipeline([('m',HistGradientBoostingClassifier(random_state=SEED))]),
 'hgb_small':Pipeline([('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,l2_regularization=1.0,random_state=SEED))]),
}
FEATS_OF={'logreg_nocode':[c for c in FEATS if c!='school_code']}
cv=GroupKFold(n_splits=5)
folds=list(cv.split(Xtr,ytr,groups=gtr))
oof={k:np.zeros(len(Xtr)) for k in MODELS}; rows=[]
for k,m in MODELS.items():
    cols=FEATS_OF.get(k,FEATS); aucs=[]
    for a,b in folds:
        f=clone(m).fit(Xtr.iloc[a][cols],ytr.iloc[a])
        p=f.predict_proba(Xtr.iloc[b][cols])[:,1]; oof[k][b]=p; aucs.append(roc_auc_score(ytr.iloc[b],p))
    rows.append((k,np.mean(aucs),np.std(aucs),log_loss(ytr,oof[k])))
cvres=pd.DataFrame(rows,columns=['model','auc_mean','auc_sd','oof_logloss']).sort_values('auc_mean',ascending=False)
cvres

## 4. Threshold for the CV winner (from out-of-fold predictions)

In [ ]:
assert cvres.iloc[0].model=='logreg_nocode'  # CV winner from the table above
best='logreg_nocode'
ths=np.linspace(0.05,0.95,91)
bal=[balanced_accuracy_score(ytr,oof[best]>=t) for t in ths]
thr=float(ths[int(np.argmax(bal))])
print('threshold',round(thr,2))

## 5. Decision (written before the test set is scored)
Chosen on cross-validation only: **`logreg_nocode`** (standardised `enrol_year`, `prior_score`, `session_minutes`, `hints_used`; logistic regression; no `school_code`), decision threshold **0.43** on the predicted pass probability. It had the best mean CV AUC (about 0.60) and the best out-of-fold log loss, and was the only candidate whose log loss beat the baseline. The gradient-boosting models and the `school_code` one-hot overfit. Nothing is changed after this point.

## 6. Refit on all training rows, score the test set once

In [ ]:
cols=FEATS_OF['logreg_nocode']
final=clone(MODELS['logreg_nocode']).fit(Xtr[cols],ytr)
pte=final.predict_proba(Xte[cols])[:,1]
pred=(pte>=0.43).astype(int)
print('test rows',len(yte),'students')
print('AUC',round(roc_auc_score(yte,pte),3),'| balanced acc',round(balanced_accuracy_score(yte,pred),3),'| F1',round(f1_score(yte,pred),3),'| log loss',round(log_loss(yte,pte),3))
print('baseline (constant) AUC 0.5, log loss',round(log_loss(yte,np.full(len(yte),ytr.mean())),3))

## 7. Conclusions
- **Headline (single test-set score, 360 held-out students, not used in any choice):** AUC 0.595, balanced accuracy 0.561, F1 0.492 at threshold 0.43; log loss 0.666 against 0.680 for the constant baseline. This matches the CV estimate (AUC about 0.60).
- **So: only weakly predictable.** The model beats chance, but it is far from useful for individual decisions. Longer sessions go with passing and more hints with failing; `prior_score`, `enrol_year` and `school_code` add almost nothing.
- Flexible models did worse in CV than the linear one, which suggests there is little non-linear signal to find in these columns.
- Caveats: (1) `session_minutes` and `hints_used` are only available once the session has run, so this is a prediction made at the end of the session, just before the quiz, not before it starts. (2) `prior_score` varies within a student, which is odd for a placement test; I have not verified how it is recorded. (3) The test set is one random split of students (no time column to split on), so its AUC has sampling error of a few hundredths.